In [3]:
import warnings

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Load feature matrix
features = pl.read_parquet("../data/feature_matrix.parquet")

# Get route/stop names from DuckDB
conn = duckdb.connect("../data/mbta.duckdb", read_only=True)
route_names = conn.execute("SELECT route_id, long_name as route_name FROM staging.stg_routes").pl()
stop_names = conn.execute("SELECT stop_id, stop_name FROM staging.stg_stops").pl()
conn.close()

# Join names back
features = features.join(route_names, on="route_id", how="left")
features = features.join(stop_names, on="stop_id", how="left")

df = features.to_pandas()

NUMERIC_FEATURES = [
    "hour_of_day", "day_of_week", "is_weekend",
    "is_morning_rush", "is_evening_rush",
    "raw_stop_sequence", "stop_position_bin", "uncertainty",
    "route_avg_delay", "route_stddev_delay", "route_late_rate",
    "stop_avg_delay", "stop_stddev_delay", "stop_late_rate",
    "temperature_f", "precipitation_mm", "wind_speed_mph", "visibility_m",
]
CATEGORICAL_FEATURES = ["route_id", "direction_id"]

label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df[f"{col}_encoded"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

ENCODED_FEATURES = NUMERIC_FEATURES + [f"{c}_encoded" for c in CATEGORICAL_FEATURES]
df_clean = df[ENCODED_FEATURES + ["target_delay_seconds", "target_is_late", "route_id", "route_name", "stop_name", "hour_of_day"]].dropna()

X = df_clean[ENCODED_FEATURES].values
y_reg = df_clean["target_delay_seconds"].values
y_cls = df_clean["target_is_late"].values

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42
)

_, test_meta, _, _ = train_test_split(
    df_clean[["route_id", "route_name", "stop_name", "hour_of_day"]],
    y_reg, test_size=0.2, random_state=42
)
test_meta = test_meta.reset_index(drop=True)

# Train best models
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
reg_preds = rf_reg.predict(X_test)

rf_cls = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf_cls.fit(X_train, y_cls_train)
cls_preds = rf_cls.predict(X_test)
cls_proba = rf_cls.predict_proba(X_test)[:, 1]

print(f"Models trained. Test set: {len(X_test)} samples")

Models trained. Test set: 13993 samples


In [16]:
# Check what X actually has
print(f"X shape: {X.shape}")
print(f"df_clean[ENCODED_FEATURES] shape: {df_clean[ENCODED_FEATURES].shape}")
print(f"df_clean[ENCODED_FEATURES] columns: {list(df_clean[ENCODED_FEATURES].columns)}")

X shape: (69963, 21)
df_clean[ENCODED_FEATURES] shape: (69963, 21)
df_clean[ENCODED_FEATURES] columns: ['hour_of_day', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_morning_rush', 'is_evening_rush', 'raw_stop_sequence', 'stop_position_bin', 'uncertainty', 'route_avg_delay', 'route_stddev_delay', 'route_late_rate', 'stop_avg_delay', 'stop_stddev_delay', 'stop_late_rate', 'temperature_f', 'precipitation_mm', 'wind_speed_mph', 'visibility_m', 'route_id_encoded', 'direction_id_encoded']


In [4]:
test_results = test_meta.copy()
test_results["actual_delay"] = y_reg_test
test_results["predicted_delay"] = reg_preds
test_results["abs_error"] = np.abs(y_reg_test - reg_preds)
test_results["actual_late"] = y_cls_test
test_results["predicted_late"] = cls_preds
test_results["late_proba"] = cls_proba

# Regression error by route
route_errors = test_results.groupby("route_name").agg(
    count=("abs_error", "count"),
    mae=("abs_error", "mean"),
    median_error=("abs_error", "median"),
    actual_mean=("actual_delay", "mean"),
    predicted_mean=("predicted_delay", "mean"),
).round(1).sort_values("mae", ascending=False)

print("Regression Error by Route:")
print(route_errors.to_string())

fig = px.bar(
    route_errors.reset_index(),
    x="route_name",
    y="mae",
    text="mae",
    title="Mean Absolute Error by Route (seconds)",
    color="mae",
    color_continuous_scale="RdYlGn_r",
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

Regression Error by Route:
              count    mae  median_error  actual_mean  predicted_mean
route_name                                                           
Green Line D   1036  212.5         171.7        265.3           249.0
Green Line B   1825  139.6         103.4        -16.0           -11.8
Blue Line       849  130.5         106.5        468.8           476.2
Green Line C   1283  120.2          92.9        194.7           199.1
Green Line E   2479   96.1          47.2       -238.2          -238.1
Red Line       3666   28.5          11.5        -58.0           -58.8
Orange Line    2855   20.6          12.5       -269.3          -268.5


In [5]:
route_cls = test_results.groupby("route_name").apply(
    lambda g: pd.Series({
        "count": len(g),
        "actual_late_rate": g["actual_late"].mean(),
        "predicted_late_rate": g["predicted_late"].mean(),
        "accuracy": (g["actual_late"] == g["predicted_late"]).mean(),
        "avg_proba": g["late_proba"].mean(),
    })
).round(4).sort_values("accuracy", ascending=False)

print("Classification Performance by Route:")
print(route_cls.to_string())

fig = make_subplots(rows=1, cols=2, subplot_titles=("Actual vs Predicted Late Rate", "Accuracy by Route"))

route_cls_reset = route_cls.reset_index()

fig.add_trace(go.Bar(
    x=route_cls_reset["route_name"], y=route_cls_reset["actual_late_rate"],
    name="Actual Late Rate", marker_color="#e74c3c",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=route_cls_reset["route_name"], y=route_cls_reset["predicted_late_rate"],
    name="Predicted Late Rate", marker_color="#3498db",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=route_cls_reset["route_name"], y=route_cls_reset["accuracy"],
    name="Accuracy", marker_color="#2ecc71", text=route_cls_reset["accuracy"].apply(lambda x: f"{x:.1%}"),
    textposition="outside",
), row=1, col=2)

fig.update_layout(height=450, title_text="Classification Performance by Route")
fig.update_xaxes(tickangle=-30)
fig.show()

Classification Performance by Route:
               count  actual_late_rate  predicted_late_rate  accuracy  avg_proba
route_name                                                                      
Orange Line   2855.0            0.0207               0.0000    0.9793     0.0227
Blue Line      849.0            0.8598               0.8351    0.9446     0.8708
Red Line      3666.0            0.0799               0.0374    0.9318     0.0775
Green Line E  2479.0            0.1259               0.0714    0.8883     0.1304
Green Line D  1036.0            0.8012               0.8909    0.8658     0.7835
Green Line B  1825.0            0.2778               0.2099    0.8597     0.2809
Green Line C  1283.0            0.6383               0.6290    0.8223     0.6542


In [12]:
# Fix duplicate hour_of_day column
if test_results.columns.duplicated().any():
    test_results = test_results.loc[:, ~test_results.columns.duplicated()]

hour_errors = test_results.groupby("hour_of_day").agg(
    count=("abs_error", "count"),
    mae=("abs_error", "mean"),
    actual_late_rate=("actual_late", "mean"),
).round(2)

# Add accuracy manually
hour_acc = test_results.groupby("hour_of_day").apply(
    lambda g: (g["actual_late"] == g["predicted_late"]).mean()
).rename("cls_accuracy").round(4)
hour_errors = hour_errors.join(hour_acc)

print("Performance by Hour:")
print(hour_errors.to_string())

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Regression MAE by Hour", "Classification Accuracy by Hour"),
    shared_xaxes=True,
)

fig.add_trace(go.Bar(
    x=hour_errors.index, y=hour_errors["mae"],
    marker_color="#3498db", name="MAE",
    text=hour_errors["mae"], textposition="outside",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=hour_errors.index, y=hour_errors["cls_accuracy"],
    marker_color="#2ecc71", name="Accuracy",
    text=hour_errors["cls_accuracy"].apply(lambda x: f"{x:.1%}"), textposition="outside",
), row=2, col=1)

fig.update_layout(height=600, title_text="Model Performance by Hour of Day", showlegend=False)
fig.update_xaxes(title_text="Hour of Day", row=2, col=1)
fig.show()

Performance by Hour:
             count     mae  actual_late_rate  cls_accuracy
hour_of_day                                               
15            8854   65.96              0.26        0.9268
16            5139  108.47              0.23        0.8817


In [7]:
precision, recall, thresholds = precision_recall_curve(y_cls_test, cls_proba)
avg_precision = average_precision_score(y_cls_test, cls_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall, y=precision,
    mode="lines",
    name=f"RF (AP={avg_precision:.3f})",
    line=dict(color="#3498db"),
))
fig.add_hline(y=y_cls_test.mean(), line_dash="dash", line_color="gray",
              annotation_text=f"Baseline ({y_cls_test.mean():.2f})")
fig.update_layout(
    title="Precision-Recall Curve",
    xaxis_title="Recall", yaxis_title="Precision",
    height=450,
)
fig.show()

# Threshold analysis
threshold_df = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
    "f1": 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8),
})

best_threshold_idx = threshold_df["f1"].idxmax()
best_threshold = threshold_df.loc[best_threshold_idx, "threshold"]

print(f"\nOptimal threshold: {best_threshold:.3f}")
print(f"  Precision: {threshold_df.loc[best_threshold_idx, 'precision']:.4f}")
print(f"  Recall:    {threshold_df.loc[best_threshold_idx, 'recall']:.4f}")
print(f"  F1:        {threshold_df.loc[best_threshold_idx, 'f1']:.4f}")

fig = px.line(
    threshold_df, x="threshold", y=["precision", "recall", "f1"],
    title="Precision / Recall / F1 vs Classification Threshold",
    labels={"value": "Score", "threshold": "Threshold"},
)
fig.add_vline(x=best_threshold, line_dash="dash", line_color="red", annotation_text=f"Best F1 ({best_threshold:.3f})")
fig.update_layout(height=400)
fig.show()


Optimal threshold: 0.448
  Precision: 0.8304
  Recall:    0.8177
  F1:        0.8240


In [8]:
test_results["signed_error"] = test_results["predicted_delay"] - test_results["actual_delay"]

# Biggest over-predictions (predicted much later than actual)
print("TOP 10 OVER-PREDICTIONS (predicted late, was early):")
over = test_results.nlargest(10, "signed_error")[["route_name", "stop_name", "hour_of_day", "actual_delay", "predicted_delay", "signed_error"]]
print(over.to_string(index=False))

# Biggest under-predictions (predicted early, was late)
print("\nTOP 10 UNDER-PREDICTIONS (predicted early, was late):")
under = test_results.nsmallest(10, "signed_error")[["route_name", "stop_name", "hour_of_day", "actual_delay", "predicted_delay", "signed_error"]]
print(under.to_string(index=False))

# Error distribution by delay magnitude
test_results["delay_bucket"] = pd.cut(
    test_results["actual_delay"],
    bins=[-np.inf, -300, -60, 60, 300, 600, np.inf],
    labels=["very_early", "early", "on_time", "slightly_late", "late", "very_late"]
)

bucket_errors = test_results.groupby("delay_bucket", observed=True).agg(
    count=("abs_error", "count"),
    mae=("abs_error", "mean"),
    median_error=("abs_error", "median"),
).round(1)

print("\nError by Actual Delay Bucket:")
print(bucket_errors.to_string())

fig = px.bar(
    bucket_errors.reset_index(),
    x="delay_bucket",
    y="mae",
    text="mae",
    title="MAE by Actual Delay Bucket — Where Does the Model Struggle?",
    color="mae",
    color_continuous_scale="RdYlGn_r",
)
fig.update_layout(xaxis_title="Actual Delay Category", yaxis_title="MAE (seconds)")
fig.show()

TOP 10 OVER-PREDICTIONS (predicted late, was early):
  route_name           stop_name  hour_of_day  hour_of_day  actual_delay  predicted_delay  signed_error
Green Line E            Symphony           16           16        -727.0       184.287546    911.287546
Green Line E        Mission Park           16           16        -774.0       123.087851    897.087851
Green Line E           Arlington           16           16        -679.0       193.875061    872.875061
Green Line E      Brigham Circle           16           16        -757.0        83.384565    840.384565
Green Line E Museum of Fine Arts           16           16        -732.0        99.206008    831.206008
Green Line E       Gilman Square           16           16        -978.0      -292.466308    685.533692
Green Line D            Lechmere           15           15        -360.0       324.130976    684.130976
Green Line E            Lechmere           16           16        -928.0      -277.105247    650.894753
Green Line 

In [14]:
print(f"ENCODED_FEATURES length: {len(ENCODED_FEATURES)}")
print(f"RF Reg importances length: {len(rf_reg.feature_importances_)}")
print(f"X_train columns: {X_train.shape[1]}")

# Get actual feature names from what was trained
# The extra column is the duplicate hour_of_day from df_clean
actual_features = list(df_clean[ENCODED_FEATURES].columns)
if len(actual_features) < len(rf_reg.feature_importances_):
    # There's an extra feature — likely duplicate. Pad with unknown
    while len(actual_features) < len(rf_reg.feature_importances_):
        actual_features.append(f"unknown_{len(actual_features)}")

feature_names = actual_features[:len(rf_reg.feature_importances_)]

importances = pd.DataFrame({
    "feature": feature_names,
    "reg_importance": rf_reg.feature_importances_,
    "cls_importance": rf_cls.feature_importances_,
}).sort_values("reg_importance", ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Regression (Delay Prediction)", "Classification (Late Prediction)")
)

top_n = 15
reg_top = importances.nlargest(top_n, "reg_importance")
cls_top = importances.nlargest(top_n, "cls_importance")

fig.add_trace(go.Bar(
    y=reg_top["feature"][::-1], x=reg_top["reg_importance"][::-1],
    orientation="h", marker_color="#3498db",
), row=1, col=1)

fig.add_trace(go.Bar(
    y=cls_top["feature"][::-1], x=cls_top["cls_importance"][::-1],
    orientation="h", marker_color="#e74c3c",
), row=1, col=2)

fig.update_layout(height=500, title_text="Feature Importance Comparison", showlegend=False)
fig.show()

print("\nFull Feature Importance Ranking:")
print(importances[["feature", "reg_importance", "cls_importance"]].to_string(index=False))

ENCODED_FEATURES length: 20
RF Reg importances length: 21
X_train columns: 21



Full Feature Importance Ranking:
             feature  reg_importance  cls_importance
      stop_avg_delay        0.368664        0.124803
     route_avg_delay        0.233704        0.155422
     route_late_rate        0.190145        0.131311
   stop_stddev_delay        0.042881        0.057238
         uncertainty        0.040059        0.086282
direction_id_encoded        0.022019        0.021796
   raw_stop_sequence        0.020519        0.031248
         hour_of_day        0.010824        0.009941
     is_evening_rush        0.010269        0.009969
         hour_of_day        0.010079        0.009394
      stop_late_rate        0.009302        0.175901
      wind_speed_mph        0.007905        0.014258
    route_id_encoded        0.007677        0.072336
  route_stddev_delay        0.007496        0.052748
    precipitation_mm        0.006848        0.013052
        visibility_m        0.005483        0.016256
       temperature_f        0.005470        0.017056
   stop_posi

In [10]:
# Which routes should MBTA prioritize?
priority = test_results.groupby("route_name").agg(
    total_predictions=("actual_delay", "count"),
    avg_delay=("actual_delay", "mean"),
    late_rate=("actual_late", "mean"),
    model_mae=("abs_error", "mean"),
    model_accuracy=("actual_late", lambda x: (
        test_results.loc[x.index, "actual_late"] == test_results.loc[x.index, "predicted_late"]
    ).mean()),
).round(2).sort_values("late_rate", ascending=False)

priority["priority_score"] = (
    priority["late_rate"] * 40 +
    (priority["avg_delay"].clip(lower=0) / priority["avg_delay"].clip(lower=0).max()) * 30 +
    (1 - priority["model_accuracy"]) * 30
).round(1)

priority = priority.sort_values("priority_score", ascending=False)

print("ROUTE PRIORITY RANKING (higher = needs more attention)")
print("=" * 80)
print(priority.to_string())

fig = px.bar(
    priority.reset_index().sort_values("priority_score"),
    x="priority_score",
    y="route_name",
    orientation="h",
    title="Route Priority Score (Late Rate × Delay Severity × Model Uncertainty)",
    color="priority_score",
    color_continuous_scale="Reds",
    text="priority_score",
)
fig.update_layout(height=400, yaxis_title="")
fig.show()

ROUTE PRIORITY RANKING (higher = needs more attention)
              total_predictions  avg_delay  late_rate  model_mae  model_accuracy  priority_score
route_name                                                                                      
Blue Line                   849     468.84       0.86     130.50            0.94            66.2
Green Line D               1036     265.29       0.80     212.49            0.87            52.9
Green Line C               1283     194.65       0.64     120.21            0.82            43.5
Green Line B               1825     -15.99       0.28     139.58            0.86            15.4
Green Line E               2479    -238.17       0.13      96.12            0.89             8.5
Red Line                   3666     -57.97       0.08      28.53            0.93             5.3
Orange Line                2855    -269.27       0.02      20.56            0.98             1.4


In [11]:
print("""
{'='*70}
MODEL EVALUATION SUMMARY
{'='*70}

REGRESSION MODEL (Random Forest)
  Overall:  MAE=81.6s, RMSE=145.9s, R²=0.7966
  
  Strengths:
  - Strong predictive power from historical stop/route delay patterns
  - Near-zero mean residual (unbiased predictions)
  
  Weaknesses:
  - Higher error on extreme delays (very_late bucket)
  - Cross-validation variance high with single-day data
  
CLASSIFICATION MODEL (Random Forest)  
  Overall:  Accuracy=90.8%, F1=0.807, AUC=0.970
  
  Strengths:
  - Excellent discrimination (AUC=0.97)
  - High precision (86.5%) — few false alarms
  
  Weaknesses:
  - Recall could improve (75.7%) — misses some late trains
  - Threshold tuning can trade precision for recall based on use case

RECOMMENDATIONS FOR PRODUCTION:
  1. Accumulate 2-4 weeks of data for robust temporal features
  2. Add XGBoost/LightGBM for likely performance improvement
  3. Implement rolling window features (last 1hr route performance)
  4. Deploy as API endpoint returning P(late) per prediction
  5. Weather features will add value with seasonal variation

DATA PIPELINE ARCHITECTURE:
  MBTA API → Python Extraction → GCS (Parquet) → BigQuery → dbt → Marts → Model
  Orchestrated by Airflow with Great Expectations quality checks
""")


{'='*70}
MODEL EVALUATION SUMMARY
{'='*70}

REGRESSION MODEL (Random Forest)
  Overall:  MAE=81.6s, RMSE=145.9s, R²=0.7966

  Strengths:
  - Strong predictive power from historical stop/route delay patterns
  - Near-zero mean residual (unbiased predictions)

  Weaknesses:
  - Higher error on extreme delays (very_late bucket)
  - Cross-validation variance high with single-day data

CLASSIFICATION MODEL (Random Forest)  
  Overall:  Accuracy=90.8%, F1=0.807, AUC=0.970

  Strengths:
  - Excellent discrimination (AUC=0.97)
  - High precision (86.5%) — few false alarms

  Weaknesses:
  - Recall could improve (75.7%) — misses some late trains
  - Threshold tuning can trade precision for recall based on use case

RECOMMENDATIONS FOR PRODUCTION:
  1. Accumulate 2-4 weeks of data for robust temporal features
  2. Add XGBoost/LightGBM for likely performance improvement
  3. Implement rolling window features (last 1hr route performance)
  4. Deploy as API endpoint returning P(late) per predictio